In [ ]:
import polars as pl
from src.features import ICARELIST, build_features, ARCHIVELIST
from src.train import train_w_folds, TEST_END, TRAIN_END, report
from dateutil.relativedelta import relativedelta
from datetime import datetime, UTC

df = pl.read_parquet("../data/parquets/sold_listings_20260901.parquet").lazy()
def train_eval(df: pl.LazyDataFrame):
    df = build_features(df)
    df = df.with_columns(
        pl.col('primary_designer').is_in(ARCHIVELIST).alias('is_archive'),
        pl.col('primary_designer').is_in(ICARELIST).alias('brands_icare')
    )
    start = datetime(2026, 2, 1, tzinfo=UTC)
    start = start - relativedelta(months=12)
    end = datetime(2026, 3, 1, tzinfo=UTC)
    return train_w_folds(df, start, end)
folds = train_eval(df)
report(folds)

In [ ]:
print('All', report(folds))
print('Archive', report(folds, 'archivelist'))
print('Same titles', report(folds, 'same_titles'))
print('Unique_titles', report(folds, 'unique_titles'))
print('Unseen_titles', report(folds, 'unseen_titles'))
print('Within 50-250', report(folds, 'within_50_250'))
print('within_50_250_and_same_titles', report(folds, 'within_50_250_and_same_titles'))